# Thesis Verification Notebook

This notebook verifies quantitative claims in the thesis that could not be verified from code alone. Each section prints the computed value alongside the thesis claim and flags any discrepancy.

**Items verified:**
1. GRU baseline parameter count (thesis: 140,804)
2. TempCNN baseline parameter count (thesis: 198,788)
3. Random Forest feature count (thesis: 104 summary statistics)
4. Training convergence values (thesis: train 0.020→0.0086, val 0.016→0.0074)
5. Table 4.10 recovery-gap sensitivity means (thesis: 6-mo=0.904, 12-mo=0.669, 24-mo=0.511)
6. Table 4.13 land-cover transition contribution (thesis: 13.4% of flagged pixels)
7. ROI area (thesis: 1,938 km²; recomputed here: 1,924 km²)

**How to run:** Runtime → Restart & Run all. Each section is independent, so partial failures don't block the rest.

In [4]:
import sys, os
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    THESIS_ROOT = '/content/drive/MyDrive/thesisv4'
else:
    THESIS_ROOT = os.path.expanduser('~/thesisv4')

sys.path.insert(0, THESIS_ROOT)

import json
import numpy as np
import pandas as pd
import config

print(f'THESIS_ROOT: {THESIS_ROOT}')
print(f'Contents of models/main:')
for f in sorted(os.listdir(os.path.join(THESIS_ROOT, 'models/main'))):
    print(f'  {f}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
THESIS_ROOT: /content/drive/MyDrive/thesisv4
Contents of models/main:
  model_main.pt
  predictions_test.npz
  results_main.json
  training_history.json


## 1. GRU baseline parameter count

Thesis claim: **140,804 parameters** (bidirectional GRU, 1 layer, 128 hidden units per direction)

In [5]:
import torch
import torch.nn as nn

# Rebuild GRUModel exactly as defined in Notebook 06b Cell 13
class GRUModel(nn.Module):
    def __init__(self, n_features, hidden_dim=128, n_layers=1, dropout=0.1):
        super().__init__()
        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
            bidirectional=True,
        )
        d_out = hidden_dim * 2
        self.dropout = nn.Dropout(dropout)
        self.burn_head         = nn.Linear(d_out, 1)
        self.severity_head     = nn.Linear(d_out, 1)
        self.persistence_head  = nn.Linear(d_out, 1)
        self.recovery_gap_head = nn.Linear(d_out, 1)

gru = GRUModel(n_features=52, hidden_dim=128, n_layers=1, dropout=0.1)
params = sum(p.numel() for p in gru.parameters())
print(f'GRU trainable parameters: {params:,}')
print(f'Thesis claim:             140,804')
if params == 140804:
    print('MATCH ✓')
else:
    print(f'MISMATCH — update thesis Table 4.7 to: {params:,}')

GRU trainable parameters: 140,804
Thesis claim:             140,804
MATCH ✓


## 2. TempCNN baseline parameter count

Thesis claim: **198,788 parameters** (3 conv blocks, kernel 5, 128 channels each)

In [6]:
# Rebuild TempCNN exactly as defined in Notebook 06b Cell 13
# (hard-coded here to avoid pulling in DEVICE and other externals from the notebook)

class TempCNNBlock(nn.Module):
    """Conv1d + BatchNorm + ReLU + Dropout block for temporal features."""
    def __init__(self, in_channels, out_channels, kernel_size=5, dropout=0.1):
        super().__init__()
        pad = kernel_size // 2
        self.conv = nn.Conv1d(in_channels, out_channels,
                              kernel_size=kernel_size, padding=pad)
        self.bn   = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.relu(self.bn(self.conv(x))))


class TempCNNModel(nn.Module):
    """Three-block TempCNN encoder with multi-task heads matching L-TAE."""
    def __init__(self, n_features, hidden_dim=128, dropout=0.1, kernel_size=5):
        super().__init__()
        self.block1 = TempCNNBlock(n_features, hidden_dim, kernel_size, dropout)
        self.block2 = TempCNNBlock(hidden_dim, hidden_dim, kernel_size, dropout)
        self.block3 = TempCNNBlock(hidden_dim, hidden_dim, kernel_size, dropout)

        self.burn_head         = nn.Conv1d(hidden_dim, 1, kernel_size=1)
        self.severity_head     = nn.Linear(hidden_dim, 1)
        self.persistence_head  = nn.Linear(hidden_dim, 1)
        self.recovery_gap_head = nn.Linear(hidden_dim, 1)


# Instantiate with default hyperparameters (n_features=52, hidden_dim=128, kernel_size=5)
tempcnn = TempCNNModel(n_features=52, hidden_dim=128, dropout=0.1, kernel_size=5)
params = sum(p.numel() for p in tempcnn.parameters())

print(f'TempCNN trainable parameters: {params:,}')
print(f'Thesis claim:                 198,788')
if params == 198788:
    print('MATCH ✓')
else:
    print(f'MISMATCH — update thesis Table 4.7 to: {params:,}')

# Per-component breakdown for transparency
print()
print("Per-component breakdown:")
for name, p in tempcnn.named_parameters():
    print(f"  {name:35s} shape={list(p.shape)!s:22s} numel={p.numel():>7,}")


TempCNN trainable parameters: 198,788
Thesis claim:                 198,788
MATCH ✓

Per-component breakdown:
  block1.conv.weight                  shape=[128, 52, 5]           numel= 33,280
  block1.conv.bias                    shape=[128]                  numel=    128
  block1.bn.weight                    shape=[128]                  numel=    128
  block1.bn.bias                      shape=[128]                  numel=    128
  block2.conv.weight                  shape=[128, 128, 5]          numel= 81,920
  block2.conv.bias                    shape=[128]                  numel=    128
  block2.bn.weight                    shape=[128]                  numel=    128
  block2.bn.bias                      shape=[128]                  numel=    128
  block3.conv.weight                  shape=[128, 128, 5]          numel= 81,920
  block3.conv.bias                    shape=[128]                  numel=    128
  block3.bn.weight                    shape=[128]                  numel=    128

## 3. Random Forest feature count

Thesis claim: **104 pixel-level summary statistics (mean, standard deviation, and quartiles per input feature)**

**Question:** 52 features × 2 stats = 104, but text says "quartiles" (which are 3 values). Which is correct?

In [7]:
# Find the RF baseline code to see exactly which statistics are computed
for cell in nb['cells']:
    if cell['cell_type'] != 'code':
        continue
    src = ''.join(cell['source'])
    if 'RandomForest' in src and ('mean' in src.lower() and 'std' in src.lower()):
        # Print statistics extraction lines
        print('=== Statistics computed in RF baseline ===')
        for line in src.split('\n'):
            if any(t in line.lower() for t in ['mean(', 'std(', 'quantile', 'percentile', 'median', 'min(', 'max(', 'np.stack', 'concat', 'features_stat']):
                print(f'  {line.strip()[:130]}')
        print()
        # Also try to find where the final feature count is printed
        for line in src.split('\n'):
            if 'shape' in line.lower() and ('rf' in line.lower() or 'stats' in line.lower() or 'feature' in line.lower()):
                print(f'  {line.strip()[:130]}')
        break

## 4. Training convergence values

Thesis claim:
- Training loss: 0.020 → 0.0086
- Validation loss: 0.016 → 0.0074
- Best validation AUC: 0.9886

In [14]:
hist_path = config.FILES['history_main']
if not os.path.isabs(hist_path):
    hist_path = os.path.join(THESIS_ROOT, hist_path)

with open(hist_path) as f:
    history = json.load(f)

print('History keys:', list(history.keys()))
print()

# Locate train / val loss under whatever key the history uses
train_key = next((k for k in history if 'train' in k.lower() and 'loss' in k.lower()), None)
val_key   = next((k for k in history if 'val'   in k.lower() and 'loss' in k.lower()), None)
auc_key   = next((k for k in history if 'auc'   in k.lower()), None)

if train_key:
    tl = history[train_key]
    print(f'Training loss ({train_key}):')
    print(f'  Epoch  1: {tl[0]:.4f}')
    print(f'  Epoch  {len(tl)}: {tl[-1]:.4f}')
    print(f'  Thesis:   0.020 → 0.0086')

if val_key:
    vl = history[val_key]
    print(f'\nValidation loss ({val_key}):')
    print(f'  Epoch  1: {vl[0]:.4f}')
    print(f'  Epoch  {len(vl)}: {vl[-1]:.4f}')
    print(f'  Thesis:   0.016 → 0.0074')

if auc_key:
    aucs = history[auc_key]
    print(f'\nValidation AUC ({auc_key}):')
    print(f'  Epoch  1: {aucs[0]:.4f}')
    print(f'  Best:     {max(aucs):.4f}')
    print(f'  Final:    {aucs[-1]:.4f}')
    print(f'  Thesis best: 0.9886')

History keys: ['train', 'val']



## 5. Table 4.10 recovery-gap sensitivity means (CRITICAL)

Thesis claim:
- 6-month window:  mean = **0.904**
- 12-month main:   mean = **0.669**
- 24-month window: mean = **0.511**

**Concern:** the mean of the analytical recovery-gap target across burned pixels in the full sample is approximately 0.502 (weighted from per-LC means in Section 4.4 Table 4.5). Table 4.10's 0.669 for the 12-month main configuration is 0.16 higher — the two numbers should be closer if they describe the same quantity.

In [9]:
# Verify by recomputing the recovery-gap target with each window
# and printing the mean over burned pixels for each configuration.

targets = np.load(config.FILES['targets_aux'])
print('targets_aux contents:', list(targets.files))

# Load the y (burned label) matrix; it should already exist for the sample
y_path = config.FILES.get('y_burned')
if y_path and not os.path.isabs(y_path):
    y_path = os.path.join(THESIS_ROOT, y_path)

y = np.load(y_path)
print(f'y shape: {y.shape}')

burned_mask = y.sum(axis=1) > 0
print(f'Burned pixels (>=1 fire month): {int(burned_mask.sum()):,}')

# The 12-month main target should be in targets_aux
rg = targets['recovery_gap']
print(f'\nRecovery-gap target from targets_aux.npz:')
print(f'  Full-sample mean: {rg.mean():.4f}')
print(f'  Burned-only mean: {rg[burned_mask].mean():.4f}')
print(f'  Thesis Table 4.10 says 12-month main: 0.669')

diff = abs(rg[burned_mask].mean() - 0.669)
if diff < 0.005:
    print(f'  MATCH ✓ (difference {diff:.4f})')
else:
    print(f'  MISMATCH — difference {diff:.4f}')
    print(f'  This means Table 4.10 was computed with a different definition')
    print(f'  or different population than the stored targets_aux.npz target.')
    print(f'  Update Table 4.10 with the correct value OR document the exact')
    print(f'  population used to compute Table 4.10.')

targets_aux contents: ['recurrence', 'severity', 'persistence', 'recovery_gap']
y shape: (608922, 108)
Burned pixels (>=1 fire month): 295,006

Recovery-gap target from targets_aux.npz:
  Full-sample mean: 0.2434
  Burned-only mean: 0.5025
  Thesis Table 4.10 says 12-month main: 0.669
  MISMATCH — difference 0.1665
  This means Table 4.10 was computed with a different definition
  or different population than the stored targets_aux.npz target.
  Update Table 4.10 with the correct value OR document the exact
  population used to compute Table 4.10.


In [10]:
# Also recompute the 6, 12, and 24-month windows directly to verify
# (only run if you have nbr_raw available)

def compute_recovery_gap(nbr_raw, y, baseline_months, recent_months):
    burned = y.sum(axis=1) > 0
    baseline_nbr = np.nanmean(
        nbr_raw[:, :baseline_months].astype(np.float64), axis=1)
    recent_nbr = np.nanmean(
        nbr_raw[:, -recent_months:].astype(np.float64), axis=1)
    gap = (baseline_nbr - recent_nbr) / (np.abs(baseline_nbr) + 1e-6)
    gap = np.clip(gap, 0, 1)
    rg = np.zeros(len(y), dtype=np.float32)
    rg[burned] = gap[burned].astype(np.float32)
    return rg

# nbr_raw must be built as in Notebook 06 Cell 15. Only run this if you
# have already produced nbr_raw in the notebook environment.
if 'nbr_raw' in dir():
    configs = [('6-month', 6, 6), ('12-month (main)', 12, 12), ('24-month', 24, 24)]
    thesis_means = {'6-month': 0.904, '12-month (main)': 0.669, '24-month': 0.511}
    for label, b, r in configs:
        rg = compute_recovery_gap(nbr_raw, y, b, r)
        m_all    = rg.mean()
        m_burned = rg[burned_mask].mean()
        print(f'{label:20s}  all_mean={m_all:.4f}  burned_mean={m_burned:.4f}  '
              f'thesis={thesis_means[label]:.4f}')
else:
    print('nbr_raw is not defined in the current notebook environment.')
    print('If you want to verify the 6/12/24-month window computation directly,')
    print('re-run Notebook 06 Cell 15 to build nbr_raw, then re-execute this cell.')
    print()
    print('For now, the previous cell (Section 5, first block) already compared the')
    print('stored targets_aux.npz recovery_gap (which is the 12-month main window)')
    print('against the thesis value 0.669.')

nbr_raw is not defined in the current notebook environment.
If you want to verify the 6/12/24-month window computation directly,
re-run Notebook 06 Cell 15 to build nbr_raw, then re-execute this cell.

For now, the previous cell (Section 5, first block) already compared the
stored targets_aux.npz recovery_gap (which is the 12-month main window)
against the thesis value 0.669.


## 6. Table 4.13 land-cover transition contribution

Thesis claim: **13.4% of flagged pixels** are on land-cover transition codes; total flagged = **260,797**.

Arithmetic check: the sum of the ten land-cover counts in Table 4.13 = 223,952. Difference = 36,845 which is 14.1% of 260,797. This differs from the claimed 13.4% by 0.7 percentage points.

In [11]:
# Sum the class-level counts as shown in the thesis Table 4.13
table_413 = {
    'cropland':            116486,
    'grassland':           49765,
    'shrubland':           16822,
    'bare':                15505,
    'forest':              10183,
    'orchards':            6651,
    'fallow':              4314,
    'covered agriculture': 3542,
    'built-up':            468,
    'water':               216,
}
sum_named = sum(table_413.values())
thesis_total = 260797
thesis_transition_pct = 13.4

print(f'Sum of named classes:  {sum_named:,}')
print(f'Thesis total flagged:  {thesis_total:,}')
print(f'Transition + other:    {thesis_total - sum_named:,}')
print(f'Transition percentage: {100 * (thesis_total - sum_named) / thesis_total:.1f}%')
print(f'Thesis claim:          {thesis_transition_pct}%')

# If you have the underlying flagged-pixels DataFrame, this is a stronger check:
flagged_path = os.path.join(THESIS_ROOT, 'outputs', 'flagged_pixels_severity_0.5.csv')
if os.path.exists(flagged_path):
    df = pd.read_csv(flagged_path)
    print(f'\nDirect check from {flagged_path}:')
    print(f'  Total flagged rows: {len(df):,}')
    if 'lc_class' in df.columns:
        vc = df['lc_class'].value_counts()
        print(f'  Distinct LC values: {len(vc)}')
        print(vc.head(20))
else:
    print(f'\n(Underlying CSV not found at {flagged_path}, skipping direct check.)')

Sum of named classes:  223,952
Thesis total flagged:  260,797
Transition + other:    36,845
Transition percentage: 14.1%
Thesis claim:          13.4%

(Underlying CSV not found at /content/drive/MyDrive/thesisv4/outputs/flagged_pixels_severity_0.5.csv, skipping direct check.)


## 7. ROI area

Thesis Section 3.2.1 states: **approximately 1,938 km²**.  
In our earlier computation of the seen/unseen fraction, an approximate value of **1,924 km²** was used.

The two numbers imply either that the exact area differs from the one assumed here, or that the value cited in the thesis needs updating.

In [12]:
# The ROI polygon is a GEE asset — check via GEE if available, otherwise
# fall back to the polygon file in the working directory.
try:
    import os, sys
    import ee
    # The project id is not committed; gee_project reads it from GEE_PROJECT
    # or from configs/gee_config.json at the repository root.
    sys.path.insert(0, os.environ.get("THESIS_ROOT", os.path.abspath("..")))
    import gee_project
    project = gee_project.require()
    ee.Initialize(project=project)
    roi_asset = f'projects/{project}/assets/gaza_envelope_for_decell_Dissolve'
    roi = ee.FeatureCollection(roi_asset)
    area_m2 = roi.geometry().area(maxError=1).getInfo()
    area_km2 = area_m2 / 1e6
    print(f'ROI area from GEE: {area_km2:.1f} km²')
    print(f'Thesis says:       1,938 km²')
    print(f'My earlier calc:   1,924 km²')
except Exception as e:
    print(f'GEE not available in this notebook: {e}')
    print()
    print('Alternative: run this in the GEE Code Editor:')
    print("  var roi = ee.FeatureCollection('gaza_envelope_for_decell_Dissolve');")
    print("  print('ROI area km²:', roi.geometry().area().divide(1e6));")

GEE not available in this notebook: Please authorize access to your Earth Engine account by running

earthengine authenticate

in your command line, or ee.Authenticate() in Python, and then retry.

Alternative: run this in the GEE Code Editor:
  var roi = ee.FeatureCollection('gaza_envelope_for_decell_Dissolve');
  print('ROI area km²:', roi.geometry().area().divide(1e6));


## 8. Summary

Copy the outputs of this notebook into a message to the code chat. Any MATCH ✓ means the thesis value is correct and needs no change; any MISMATCH means the thesis text (or table entry) needs to be updated to the computed value, or the discrepancy needs to be documented.